In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window

spark = (
    SparkSession.builder
    .appName("Customer Orders Window")
    .getOrCreate()
)

df = spark.read.csv(
    "orders.csv",
    header=True,
    inferSchema=True
)

df = df.withColumn(
    "placed_at",
    F.to_timestamp(F.col("placed_at"))
)

df = df.withColumn(
    "revenue",
    F.col("quantity") * F.col("price_each")
)

customer_window = Window.partitionBy("customer_id").orderBy("placed_at")

result = (
    df.withColumn("seq", F.row_number().over(customer_window))
      .withColumn("prev_amt", F.lag("revenue").over(customer_window))
      .withColumn("prev_date", F.lag("placed_at").over(customer_window))
      .withColumn("days_since_prev", F.datediff(F.col("placed_at"), F.col("prev_date")))
)

result.filter(F.col("customer_id") == 10).show()